# 18 — Hybrid Field Merge (Baseline + Improved)

This notebook defines a deterministic merge policy for invoice extraction outputs from two pipelines:
- `baseline`
- `improved`

It includes:
1. `is_empty(val)` helper
2. `is_ocr_noisy(val)` helper
3. `merge_extractions(baseline, improved)`
4. Unit tests
5. Demo runs on test-set images and custom image paths


In [25]:
import re
import json
import sys
from pathlib import Path
from typing import Any, Dict

import torch
from PIL import Image
from datasets import load_from_disk
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
EXTRACTOR_CKPT = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from invoice_cleaner import InvoiceCleaner
from extraction_improvements import process_invoice as improved_process_invoice

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)

print(f'Project root : {PROJECT_ROOT}')
print(f'Device       : {DEVICE}')
print(f'Extractor ckpt exists: {EXTRACTOR_CKPT.exists()}')


Project root : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification
Device       : mps
Extractor ckpt exists: True


In [26]:
import re
from typing import Any, Dict, List, Optional, Tuple

# Canonical output field order (uppercase, as requested).
FIELDS = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME', 'RECIPIENT_NAME', 'TOTAL_AMOUNT',
]

LOWER_TO_UPPER = {
    'invoice_number': 'INVOICE_NUMBER',
    'invoice_date': 'INVOICE_DATE',
    'due_date': 'DUE_DATE',
    'issuer_name': 'ISSUER_NAME',
    'recipient_name': 'RECIPIENT_NAME',
    'total_amount': 'TOTAL_AMOUNT',
}

_LABEL_ONLY_INVOICE = {'INVOICE', 'ID', 'BUYER'}
_RECIPIENT_LABEL_RE = re.compile(r'\b(?:INVOICE|SHIP\s+TO|BILL\s+TO|FROM|TO)\b', re.IGNORECASE)
_PHONE_RE = re.compile(r'\+?\d[\d\-\(\)\s]{7,}')
_ADDRESS_WORD_RE = re.compile(r'\b(?:Street|St\.|Ave|Road|Drive|Lane|Court|Square)\b', re.IGNORECASE)
_STATE_ZIP_RE = re.compile(r',\s*[A-Z]{2}\s+\d{5}')

# Strong invoice patterns for OCR fallback.
_INVOICE_CANDIDATE_RE = re.compile(r'^[A-Z]{2,}[\-/]?\d{4,}$')
_YEAR_OFFSET_RE = re.compile(r'^\dY\dM\dd-\d+$', re.IGNORECASE)


def is_empty(val: Any) -> bool:
    """True for None, empty string, or em-dash sentinel."""
    if val is None:
        return True
    if isinstance(val, str):
        s = val.strip()
        return s == '' or s == '—'
    return False


def is_ocr_noisy(val: Any) -> bool:
    """Heuristic OCR-noise detector for recipient names."""
    if is_empty(val):
        return False

    s = str(val).strip()

    # Existing checks.
    if '!' in s:
        return True
    if re.search(r'[A-Za-z]', s) and re.search(r'\d', s):
        return True
    if len(s) > 40:
        return True

    # Added checks.
    if _RECIPIENT_LABEL_RE.search(s):
        return True
    if '#' in s:
        return True
    if _PHONE_RE.search(s):
        return True
    if _ADDRESS_WORD_RE.search(s):
        return True
    if _STATE_ZIP_RE.search(s):
        return True

    return False


def _clean_noisy_recipient(val: Any) -> str:
    """Remove known noise patterns from recipient text."""
    if is_empty(val):
        return ''

    s = str(val)

    # Remove noisy structures using same patterns used for noise detection.
    s = _PHONE_RE.sub(' ', s)
    s = _STATE_ZIP_RE.sub(' ', s)
    s = _ADDRESS_WORD_RE.sub(' ', s)
    s = _RECIPIENT_LABEL_RE.sub(' ', s)
    s = s.replace('#', ' ')

    # Remove common OCR code-like fragments often glued to labels.
    s = re.sub(r'\b[A-Z]{2,}-\d{2,}\b', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip(' ,.-')

    # Keep a name-like tail if present (2+ title-case-ish tokens).
    name_chunks = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+', s)
    if name_chunks:
        return name_chunks[-1].strip()

    return s


def _dedup_name(s: str) -> str:
    """Remove repeated name fragments: 'John Smith John Smith' -> 'John Smith'."""
    tokens = s.split()
    half = len(tokens) // 2
    if half > 0 and tokens[:half] == tokens[half:half * 2]:
        return ' '.join(tokens[:half])
    return s


def _to_upper_fields(d: Dict[str, Any]) -> Dict[str, Any]:
    """Accept either uppercase or lowercase-key dict and normalize to uppercase."""
    out: Dict[str, Any] = {k: None for k in FIELDS}
    for k, v in (d or {}).items():
        if k in out:
            out[k] = v
        elif k in LOWER_TO_UPPER:
            out[LOWER_TO_UPPER[k]] = v
    return out


def _pick_invoice_candidate(ocr_candidates: Optional[List[str]]) -> str:
    if not ocr_candidates:
        return ''
    for cand in ocr_candidates:
        c = str(cand).strip()
        if _INVOICE_CANDIDATE_RE.match(c) or _YEAR_OFFSET_RE.match(c):
            return c
    return ''


def merge_extractions(
    baseline: dict,
    improved: dict,
    ocr_candidates: Optional[List[str]] = None,
) -> dict:
    """
    Build hybrid extraction output using fixed per-field merge rules.
    Returns uppercase keys only.
    """
    b = _to_upper_fields(baseline or {})
    i = _to_upper_fields(improved or {})

    merged: Dict[str, Any] = {}

    # Rule: value-like fields are taken from improved first; baseline is backup.
    for field in ('INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE', 'TOTAL_AMOUNT'):
        merged[field] = i[field] if not is_empty(i[field]) else b[field]

    # Rule: issuer fallback from improved only when baseline is empty (baseline is usually safer).
    merged['ISSUER_NAME'] = b['ISSUER_NAME'] if not is_empty(b['ISSUER_NAME']) else i['ISSUER_NAME']

    # Rule: recipient uses baseline unless baseline looks OCR-noisy.
    base_rec = b['RECIPIENT_NAME']
    imp_rec = i['RECIPIENT_NAME']
    if is_ocr_noisy(base_rec):
        if not is_empty(imp_rec) and not is_ocr_noisy(imp_rec):
            merged['RECIPIENT_NAME'] = imp_rec
        else:
            cleaned = _clean_noisy_recipient(base_rec)
            merged['RECIPIENT_NAME'] = cleaned if not is_empty(cleaned) else base_rec
    else:
        merged['RECIPIENT_NAME'] = base_rec if not is_empty(base_rec) else imp_rec

    # Optional cleanup: collapse accidental duplicate full-name repeats from OCR/merge.
    if not is_empty(merged.get('RECIPIENT_NAME')):
        merged['RECIPIENT_NAME'] = _dedup_name(str(merged['RECIPIENT_NAME']).strip())

    # Fix: amount decimal normalization for currency-prefix comma decimals (e.g., £55,00).
    amt = '' if is_empty(merged['TOTAL_AMOUNT']) else str(merged['TOTAL_AMOUNT']).strip()
    if re.match(r'^[£$€]\d+,\d{2}$', amt):
        merged['TOTAL_AMOUNT'] = amt.replace(',', '.', 1)

    # Fix: reject label-only invoice numbers, then OCR fallback.
    inv = '' if is_empty(merged['INVOICE_NUMBER']) else str(merged['INVOICE_NUMBER']).strip()
    if inv.upper() in _LABEL_ONLY_INVOICE and not re.search(r'\d', inv):
        merged['INVOICE_NUMBER'] = '—'
        fallback = _pick_invoice_candidate(ocr_candidates)
        if fallback:
            merged['INVOICE_NUMBER'] = fallback

    # Fix: issuer/recipient swap suspicion guard.
    issuer = '' if is_empty(merged['ISSUER_NAME']) else str(merged['ISSUER_NAME']).strip()
    recipient = '' if is_empty(merged['RECIPIENT_NAME']) else str(merged['RECIPIENT_NAME']).strip()
    if issuer and recipient and issuer.lower() == recipient.lower():
        merged['_name_swap_suspected'] = True
        merged['ISSUER_NAME'] = '—'

    return merged


def _merge_with_source(
    baseline: dict,
    improved: dict,
    ocr_candidates: Optional[List[str]] = None,
) -> Tuple[dict, dict]:
    """Internal merge that also returns per-field source labels."""
    b = _to_upper_fields(baseline or {})
    i = _to_upper_fields(improved or {})

    out = merge_extractions(baseline, improved, ocr_candidates=ocr_candidates)
    source: Dict[str, str] = {}

    for field in ('INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE', 'TOTAL_AMOUNT'):
        if is_empty(i[field]) and not is_empty(b[field]):
            expected = b[field]
            primary = 'baseline'
        elif not is_empty(i[field]):
            expected = i[field]
            primary = 'improved'
        else:
            expected = i[field]
            primary = 'improved'

        source[field] = primary if str(out.get(field, '')) == str(expected) else 'cleaned'

    # ISSUER_NAME base choice logic.
    issuer_expected = b['ISSUER_NAME'] if not is_empty(b['ISSUER_NAME']) else i['ISSUER_NAME']
    source['ISSUER_NAME'] = 'baseline' if issuer_expected == b['ISSUER_NAME'] else 'improved'
    if str(out.get('ISSUER_NAME', '')) != str(issuer_expected):
        source['ISSUER_NAME'] = 'cleaned'

    # RECIPIENT_NAME base choice logic with noise handling.
    if is_ocr_noisy(b['RECIPIENT_NAME']):
        if not is_empty(i['RECIPIENT_NAME']) and not is_ocr_noisy(i['RECIPIENT_NAME']):
            rec_expected = i['RECIPIENT_NAME']
            rec_primary = 'improved'
        else:
            rec_expected = _clean_noisy_recipient(b['RECIPIENT_NAME']) or b['RECIPIENT_NAME']
            rec_primary = 'cleaned'
    else:
        rec_expected = b['RECIPIENT_NAME'] if not is_empty(b['RECIPIENT_NAME']) else i['RECIPIENT_NAME']
        rec_primary = 'baseline' if not is_empty(b['RECIPIENT_NAME']) else 'improved'

    source['RECIPIENT_NAME'] = rec_primary if str(out.get('RECIPIENT_NAME', '')) == str(rec_expected) else 'cleaned'

    return out, source


def merge_batch(records: List[dict]) -> List[dict]:
    """
    Merge many records.
    Each record should contain: {'baseline': {...}, 'improved': {...}, 'ocr_candidates': [...](optional)}.
    Adds '_source' with per-field winner: baseline / improved / cleaned.
    """
    merged_records: List[dict] = []
    for rec in records:
        out, source = _merge_with_source(
            rec.get('baseline', {}),
            rec.get('improved', {}),
            rec.get('ocr_candidates', None),
        )
        out['_source'] = source
        merged_records.append(out)
    return merged_records


print('merge_extractions + merge_batch helpers defined OK')


merge_extractions + merge_batch helpers defined OK


In [27]:
# Unit tests for targeted fixes

def _assert_equal(a, b, msg):
    if a != b:
        raise AssertionError(f"{msg}\nExpected: {b}\nActual:   {a}")

# 1) Recipient noisy baseline cleaned fallback
b1 = {'RECIPIENT_NAME': 'SHIP TO INVOICE # US-001 John Smith'}
i1 = {'RECIPIENT_NAME': '—'}
m1 = merge_extractions(b1, i1)
_assert_equal(m1['RECIPIENT_NAME'], 'John Smith', 'Test1 recipient cleaned fallback')

# 2) Issuer/recipient swap suspected
b2 = {'ISSUER_NAME': 'John Smith', 'RECIPIENT_NAME': 'John Smith'}
i2 = {'ISSUER_NAME': 'John Smith', 'RECIPIENT_NAME': 'John Smith'}
m2 = merge_extractions(b2, i2)
_assert_equal(m2['ISSUER_NAME'], '—', 'Test2 issuer should be cleared on swap suspicion')
_assert_equal(m2.get('_name_swap_suspected', False), True, 'Test2 swap flag')

# 3) Amount decimal normalization
b3 = {'TOTAL_AMOUNT': '—'}
i3 = {'TOTAL_AMOUNT': '£55,00'}
m3 = merge_extractions(b3, i3)
_assert_equal(m3['TOTAL_AMOUNT'], '£55.00', 'Test3 decimal normalization')

# 4) Label-only invoice number with OCR fallback
b4 = {'INVOICE_NUMBER': '—'}
i4 = {'INVOICE_NUMBER': 'INVOICE'}
m4 = merge_extractions(b4, i4, ocr_candidates=['abc', 'INV15261847'])
_assert_equal(m4['INVOICE_NUMBER'], 'INV15261847', 'Test4 invoice OCR fallback')

# 5) Clean record pass-through + merge_batch _source correctness
clean_base = {
    'INVOICE_NUMBER': 'INV-7788',
    'INVOICE_DATE': '03-Mar-2023',
    'DUE_DATE': '10-Mar-2023',
    'ISSUER_NAME': 'Alpha Ltd',
    'RECIPIENT_NAME': 'Maria Costa',
    'TOTAL_AMOUNT': '55.00 EUR',
}
clean_impr = dict(clean_base)
batch_out = merge_batch([{'baseline': clean_base, 'improved': clean_impr}])[0]
for k, v in clean_base.items():
    _assert_equal(batch_out[k], v, f'Test5 pass-through field {k}')
_assert_equal(batch_out['_source'], {
    'INVOICE_NUMBER': 'improved',
    'INVOICE_DATE': 'improved',
    'DUE_DATE': 'improved',
    'TOTAL_AMOUNT': 'improved',
    'ISSUER_NAME': 'baseline',
    'RECIPIENT_NAME': 'baseline',
}, 'Test5 _source correctness')

print('All targeted unit tests passed (5/5).')


All targeted unit tests passed (5/5).


## Pipeline Setup (Baseline + Improved)

This section loads the extractor and defines:
- `baseline_pipeline(image_path)` (Notebook 13 style)
- `improved_pipeline(image_path)` (from `src/extraction_improvements.py`)


In [28]:
# Load label mapping + dataset
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

raw_dataset = load_from_disk(str(DATASET_DIR))

# Load extractor
processor = LayoutLMv3Processor.from_pretrained(
    str(EXTRACTOR_CKPT), local_files_only=True, use_fast=True
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(EXTRACTOR_CKPT), local_files_only=True
)
model.to(DEVICE)
model.eval()

cleaner = InvoiceCleaner()

print('Dataset loaded       :', raw_dataset)
print('Extractor loaded OK  :', EXTRACTOR_CKPT)


Loading weights: 100%|██████████| 216/216 [00:00<00:00, 7151.81it/s]


Dataset loaded       : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})
Extractor loaded OK  : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint


In [29]:
def _baseline_get_raw(image, words, bboxes):
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=512, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})

    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids = encoding.word_ids(batch_index=0)
    word_preds = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    aligned_words = [words[i] for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]

    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
                current_field, current_tokens = fn, [word]

    if current_field:
        t = ' '.join(current_tokens).strip()
        if t:
            fields[current_field] = t

    return fields

def baseline_pipeline(image_path: str) -> Dict[str, Any]:
    import pytesseract

    image = Image.open(image_path).convert('RGB')
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    words, boxes = [], []
    w, h = image.size
    for i, text in enumerate(data['text']):
        text = str(text).strip()
        if not text or int(data['conf'][i]) < 10:
            continue
        left, top = data['left'][i], data['top'][i]
        width, height = data['width'][i], data['height'][i]

        x0 = int(max(0, min(1000, round(left / w * 1000))))
        y0 = int(max(0, min(1000, round(top / h * 1000))))
        x1 = int(max(0, min(1000, round((left + width) / w * 1000))))
        y1 = int(max(0, min(1000, round((top + height) / h * 1000))))
        if x1 <= x0:
            x1 = min(1000, x0 + 1)
        if y1 <= y0:
            y1 = min(1000, y0 + 1)

        words.append(text)
        boxes.append([x0, y0, x1, y1])

    words = words or ['empty']
    boxes = boxes or [[0, 0, 1, 1]]

    raw = _baseline_get_raw(image, words, boxes)
    return cleaner.clean(raw, ocr_words=words)

def improved_pipeline(image_path: str) -> Dict[str, Any]:
    return improved_process_invoice(
        image_path,
        extractor_model=model,
        extractor_processor=processor,
        device=DEVICE,
        id2label=id2label,
        cleaner=cleaner,
        classifier_model=None,
        classifier_processor=None,
        ocr_engine='doctr',
        confidence_threshold=0.70,
        max_length=512,
    )

print('baseline_pipeline() and improved_pipeline() defined OK')


baseline_pipeline() and improved_pipeline() defined OK


## Demo 1: Test-Set Images

Runs baseline, improved, and merged outputs side-by-side on a few images from `raw_dataset['test']`.


In [30]:
N_TEST_IMAGES = 5

for i in range(N_TEST_IMAGES):
    example = raw_dataset['test'][i]
    img_path = example['image_path']
    stem = Path(img_path).stem

    base = baseline_pipeline(img_path)
    impr = improved_pipeline(img_path)
    merged = merge_extractions(base, impr)

    print('\n' + '=' * 95)
    print(f'  TEST {i}: {stem}')
    print('=' * 95)
    print(f"  {'FIELD':<18}  {'BASELINE':<28}  {'IMPROVED':<28}  MERGED")
    print(f"  {'-'*18}  {'-'*28}  {'-'*28}  {'-'*28}")

    base_u = _to_upper_fields(base)
    impr_u = _to_upper_fields(impr)
    for field in FIELDS:
        b = base_u.get(field) or '—'
        m = impr_u.get(field) or '—'
        h = merged.get(field) or '—'
        b_d = (str(b)[:26] + '..') if len(str(b)) > 28 else str(b)
        m_d = (str(m)[:26] + '..') if len(str(m)) > 28 else str(m)
        h_d = (str(h)[:26] + '..') if len(str(h)) > 28 else str(h)
        print(f"  {field:<18}  {b_d:<28}  {m_d:<28}  {h_d}")



  TEST 0: Template1_Instance189
  FIELD               BASELINE                      IMPROVED                      MERGED
  ------------------  ----------------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER      —                             —                             —
  INVOICE_DATE        29-Apr-2012                   29-Apr-2012                   29-Apr-2012
  DUE_DATE            07-Aug-2010                   07-Aug-2010                   07-Aug-2010
  ISSUER_NAME         Mission                       —                             Mission
  RECIPIENT_NAME      Shelly Rodriguez              Shelly Rodriguez Markville..  Shelly Rodriguez
  TOTAL_AMOUNT        134.41 USD                    134.41 USD                    134.41 USD

  TEST 1: Template38_Instance29
  FIELD               BASELINE                      IMPROVED                      MERGED
  ------------------  ----------------------------  ----------------------------  ---------

## Demo 2: Custom Image Paths

Put your own image/PDF paths below and run this cell.


In [31]:
CUSTOM_IMAGE_PATHS = [
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_1.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_2.avif',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_3.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_4.png',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_5.png',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_6.png',
]

for p in CUSTOM_IMAGE_PATHS:
    path = Path(p)
    print('\n' + '=' * 95)
    print(f'  FILE: {path.name}')
    print('=' * 95)

    if not path.exists():
        print('  File not found, skipping.')
        continue

    try:
        base = baseline_pipeline(str(path))
        impr = improved_pipeline(str(path))
        merged = merge_extractions(base, impr)

        base_u = _to_upper_fields(base)
        impr_u = _to_upper_fields(impr)

        print(f"  {'FIELD':<18}  {'BASELINE':<28}  {'IMPROVED':<28}  MERGED")
        print(f"  {'-'*18}  {'-'*28}  {'-'*28}  {'-'*28}")

        for field in FIELDS:
            b = base_u.get(field) or '—'
            m = impr_u.get(field) or '—'
            h = merged.get(field) or '—'
            b_d = (str(b)[:26] + '..') if len(str(b)) > 28 else str(b)
            m_d = (str(m)[:26] + '..') if len(str(m)) > 28 else str(m)
            h_d = (str(h)[:26] + '..') if len(str(h)) > 28 else str(h)
            print(f"  {field:<18}  {b_d:<28}  {m_d:<28}  {h_d}")
    except Exception as e:
        print(f'  Error: {e.__class__.__name__}: {e}')



  FILE: doc_i_1.webp
  FIELD               BASELINE                      IMPROVED                      MERGED
  ------------------  ----------------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER      12345                         12345                         12345
  INVOICE_DATE        —                             —                             —
  DUE_DATE            —                             —                             —
  ISSUER_NAME         Imani Olowe                   Imani Olowe                   —
  RECIPIENT_NAME      Imani Olowe +123-456-7890     Imani Olowe +123-456-7890     Imani Olowe
  TOTAL_AMOUNT        $2750                         $2750                         $2750

  FILE: doc_i_2.avif
  FIELD               BASELINE                      IMPROVED                      MERGED
  ------------------  ----------------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER      12345       